<a href="https://colab.research.google.com/github/marisolriveraslrzn/CodingIA/blob/main/Text_generation_fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Text generation - Fine-tuning


In [ ]:
!pip install transformers datasets

In [ ]:
from transformers import AutoTokenizer, GPT2LMHeadModel, pipeline, Trainer, TrainingArguments
import pandas as pd
from datasets import Dataset

model_name = "gpt2"

# Tokenizer shared by both models
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Model before training
model_orig = GPT2LMHeadModel.from_pretrained(model_name)
model_orig.resize_token_embeddings(len(tokenizer))

# Model that we will train (can fine-tune later)
model_tuned = GPT2LMHeadModel.from_pretrained(model_name)
model_tuned.resize_token_embeddings(len(tokenizer))

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Embedding(50257, 768)

In [ ]:
generator_orig = pipeline("text-generation", model=model_orig, tokenizer=tokenizer)

prompt = "Game: Stray\nDescription:"
result = generator_orig(
    prompt,
    max_new_tokens=100,
    num_return_sequences=1,
    do_sample=True,
    temperature=0.9)
print("Text BEFORE training:")
print(result[0]["generated_text"])

[transformers] Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'do_sample', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Text BEFORE training:
Game: Stray
Description: A character named "Girly," the main character of the short story that is set against the backdrop of the Second World War. G.H. is a veteran officer hired by a merchant organization to assist a company to liberate a prisoner camp from the Nazis, which he hopes will serve as a means of settling relations between the Nazi occupation and other parts of the Allied Empire. The character has an exceptional ability to walk, walk up and down walls, and move in, down, and over anything


In [ ]:
#Loading data and creating a dataset
with open("game.txt", "r", encoding="utf-8") as f:
    text = f.read()

texts = [blok.strip() for blok in text.split("===") if blok.strip()]

df = pd.DataFrame({"text": texts})
dataset = Dataset.from_pandas(df)

In [ ]:
#Tokenization

tokenizer.pad_token = tokenizer.eos_token

def tokenize(batch):
    tokens = tokenizer(batch["text"], padding=True, truncation=True, max_length=512)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [ ]:
#Fine-tuning the model

training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    num_train_epochs=10,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    logging_steps=2,
    save_steps=1000,
    save_total_limit=1,
    report_to="none",
)

trainer = Trainer(
    model=model_tuned,
    args=training_args,
    train_dataset=tokenized_dataset,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
2,7.282208
4,5.753758
6,5.122536
8,3.996162
10,4.353151
12,3.325219
14,3.788639
16,3.750496
18,3.604858
20,3.362802


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=20, training_loss=4.433983063697815, metrics={'train_runtime': 72.3433, 'train_samples_per_second': 0.691, 'train_steps_per_second': 0.276, 'total_flos': 1888243200000.0, 'train_loss': 4.433983063697815, 'epoch': 10.0})

In [ ]:
#Generando una nueva descripción del juego después del entrenamiento

generator_tuned = pipeline("text-generation", model=model_tuned, tokenizer=tokenizer)

prompt = "Game: Stray\nDescription:"
result = generator_tuned(
    prompt,
    max_new_tokens=100,
    num_return_sequences=1,
    do_sample=True,
    temperature=0.8,
    num_beams=3
    )
print("Text AFTER training:")
print(result[0]["generated_text"])

[transformers] Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'do_sample', 'temperature', 'max_new_tokens', 'num_beams'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Text AFTER training:
Game: Stray
Description: Stray is a game in which you play as a young boy who finds himself trapped in a mysterious world. As you play as a boy trapped in a world where you must navigate your way through a series of puzzles, puzzles, and puzzles to find your way through a world of puzzles, puzzles, puzzles, and puzzles to find your way through the world. The game is set in a world of puzzles, puzzles, and puzzles to solve. The game is set in a world where you play as
